## Import Libraries

In [1]:
!pip install qiskit --upgrade
!pip install qiskit-aer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.8/6.8 MB 28.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.4/119.4 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 15.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.5/49.5 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 MB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.0/109.0 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 69.0 MB/s eta 0:00:00


In [2]:

import numpy as np
from qiskit import QuantumCircuit
from qiskit.quantum_info import Kraus, SuperOp
from qiskit.visualization import plot_histogram
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit.quantum_info import Statevector,DensityMatrix,state_fidelity,partial_trace, Operator
from matplotlib import pyplot as plt
from functools import reduce
from scipy.linalg import expm
import pandas as pd
from qiskit_aer import AerSimulator
from qiskit import transpile

#from qiskit import Aer, execute, transpile


# Import from Qiskit Aer noise module
from qiskit_aer.noise import (
    NoiseModel,
    QuantumError,
    ReadoutError,
    depolarizing_error,
    pauli_error,
    thermal_relaxation_error,
)

Hamiltonian:

$H=-h(X_1+X_2)-J Z_1 Z_2$

Trotter formula:

$e^{-iH t}=(e^{i h X_1 \Delta t}e^{i h X_2 \Delta t}e^{i J Z_1 Z_2 \Delta t})^{N_{\text{Trot}}},$

where $\Delta t=t/N_{\text{Trot}}$.

## AER Pure SWAPNET

### Set Base Funcs for base state with noise

In [3]:
def circuit_with_noise_at_end(qc, d, epsilon):
    # Define the depolarizing error
    depolarizing = depolarizing_error(epsilon, d)

    # Apply the depolarizing noise to each d-sized register
    qc.append(depolarizing, range(d, 2*d))  # Second register
    qc.append(depolarizing, range(2*d, 3*d))  # Third register
    qc.append(depolarizing, range(3*d, 4*d))  # Fourth register

    return qc



def getExactState(n_qubits):
    # Create an initial state (|00...0>)
    initial_state = np.zeros(2**n_qubits)
    #print('Initial array:', initial_state)
    initial_state[0] = 1
    pure_state = DensityMatrix(initial_state)

    return pure_state

# Input state without purification, used for benchmarking

def getInputRho(epsilon, d):

    # Create a Quantum Circuit acting on a quantum register of d qubits

    depolarizing = depolarizing_error(epsilon, d)
    qc = QuantumCircuit(d)
    qc.append(depolarizing, [0, d-1])
    state = DensityMatrix(qc)

    return state

### Set SWAPNET

In [4]:
def get_noisy_sampled(d, epsilon):

    # Select Aer Simulator backend
    simulator = AerSimulator()

    def execute_circuit_on_state(qc):
        """ Executes a circuit on the AerSimulator and returns the state result. """
        qc_transpiled = transpile(qc, simulator)
        result = simulator.run(qc_transpiled).result()
        return result.get_statevector(qc_transpiled)


    # Initialize quantum circuit with classical registers
    qc = QuantumCircuit(4 * d)  # Extra qubit and classical bit for parity check

    # Apply noise (assuming function exists)
    qc = circuit_with_noise_at_end(qc, d, epsilon)

    qc.save_statevector()
    state = execute_circuit_on_state(qc)

    return state

def QPA_Result_with_AER(d, epsilon, N):

    # Select Aer Simulator backend
    simulator = AerSimulator()

    def execute_circuit_on_state(qc):
        """ Executes a circuit on the AerSimulator and returns the state result. """
        qc_transpiled = transpile(qc, simulator)
        result = simulator.run(qc_transpiled).result()
        return result.get_statevector(qc_transpiled)

    # Define qc_odd and qc_even (acting on quantum bits only)
    qc_odd = QuantumCircuit(4 * d)
    qc_even = QuantumCircuit(4 * d)

    # Apply Hadamard and CNOTs for GHZ state in both circuits
    for k in range(d):
        qc_odd.reset(k)
        qc_even.reset(k)

    qc_odd.h(0)
    qc_even.h(0)

    for k in range(d - 1):
        qc_odd.cx(0, 1 + k)
        qc_even.cx(0, 1 + k)

    # Apply CSWAPs and Hadamards for swapping operations
    for k in range(d):
        qc_odd.cswap(0 + k, d + k, 2 * d + k)  # SYM12 swap
        qc_even.cswap(0 + k, d + k, 3 * d + k)  # SYM13 swap

    for k in range(d):
        qc_odd.h(k)
        qc_even.h(k)

    # Initialize quantum circuit with classical registers
    qc = QuantumCircuit(4 * d + 1, d + 1)  # Extra qubit and classical bit for parity check

    # Apply noise (assuming function exists)
    qc = circuit_with_noise_at_end(qc, d, epsilon)

    qc.h(0)#q0_firstqbit = |0>+|1>/sqrt2
    for k in range(d-1):
      qc.cx(0,1+k)#q0 = |0000...> + |1111...>/sqrt2


    # Apply the first CSWAP gate controlled by q0, targeting q1 and q2
    for k in range(d):
      qc.cswap(0+k, k+d, k+2*d)#|+>_k x SYM12_k + |->_k x AntiSYM12_k /norm

    # Apply the second Hadamard gate to q0
    for k in range(d):
      qc.h(k) #|0> x SYM12 + |1> x AntiSYM12 /norm
    # Measure qubits 0 to d-1 into classical bits 0 to d-1
    for i in range(d):
        qc.measure(i, i)

    # **Compute parity classically**: If even, flip an auxiliary qubit (qubit index `4*d`)
    for i in range(1, d):  # XOR all classical bits and store in classical bit `d`
        qc.x(4 * d).c_if(i, 1)

    qc.measure(4 * d, d)  # Measure parity result into classical bit `d`

    # Apply `qc_odd` and `qc_even` **if parity bit is 0** (i.e., even number of 1s)
    qc.append(qc_odd.to_instruction(), qargs=qc.qubits[:4*d]).c_if(qc.clbits[d], 0)
    qc.append(qc_even.to_instruction(), qargs=qc.qubits[:4*d]).c_if(qc.clbits[d], 0)


    qc.save_statevector()
    state = execute_circuit_on_state(qc)

    return state

# Run the function
d=2
epsilon = 0.0
N_qpa = 1
psi_purified = QPA_Result_with_AER(d, epsilon, N_qpa)
psi_base = get_noisy_sampled(d, epsilon)
psi_purified.draw('latex')
#psi_base.draw('latex')



<ipython-input-4-f153bb4300ae>:84: DeprecationWarning: The method ``qiskit.circuit.instructionset.InstructionSet.c_if()`` is deprecated as of qiskit 1.3.0. It will be removed in 2.0.0.
  qc.x(4 * d).c_if(i, 1)
<ipython-input-4-f153bb4300ae>:89: DeprecationWarning: The method ``qiskit.circuit.instructionset.InstructionSet.c_if()`` is deprecated as of qiskit 1.3.0. It will be removed in 2.0.0.
  qc.append(qc_odd.to_instruction(), qargs=qc.qubits[:4*d]).c_if(qc.clbits[d], 0)
<ipython-input-4-f153bb4300ae>:90: DeprecationWarning: The method ``qiskit.circuit.instructionset.InstructionSet.c_if()`` is deprecated as of qiskit 1.3.0. It will be removed in 2.0.0.
  qc.append(qc_even.to_instruction(), qargs=qc.qubits[:4*d]).c_if(qc.clbits[d], 0)


<IPython.core.display.Latex object>